# Regret Signals Whether to Search More Broadly or More Deeply

- Rio Sumida, Hidezo Suganuma, Ryutaro Mori, Yukiko Muramoto, and Tatsuya Kameda

Simulation code for the computational model (Figure 2). Run all cells from top to bottom; the simulation cell generates the data file that the figure cell reads.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import beta
from scipy.stats import gamma
from tqdm import tqdm
import os
import matplotlib.colors as mcolors


In [ ]:
class Environment:
    def __init__(self, k: int, type: str, *paremeters, seed: int = None):
        self.k = k
        if len(paremeters) != 2:
            raise ValueError('Invalid parameters')
        self.parameters = paremeters
        self.type = type
        
        if seed is not None:
            np.random.seed(seed)
        
        if type == 'beta':
            self.mu = np.random.beta(self.parameters[0], self.parameters[1], self.k)
        elif type == 'gamma':
            self.mu = np.random.gamma(self.parameters[0], self.parameters[1], self.k)
        else:
            raise ValueError('Invalid type')
    
    def pull(self, arm: int):
        if arm < 0 or arm >= self.k:
            raise ValueError('Invalid arm')
        return self.mu[arm]
    
    def plot_distribution(self):
        fig, axes = plt.subplots(1, 2, figsize=(8, 4))
        if self.type == 'beta':
            # plot beta distribution
            x = np.linspace(0, 1, 100)
            y = beta.pdf(x, self.parameters[0], self.parameters[1])
            axes[0].plot(x, y)
            axes[0].set_ylabel('Density')
            # plot hist of mu
            axes[1].hist(self.mu)
            axes[1].set_xlim(0, 1)
            axes[1].set_ylabel('Frequency')
        elif self.type == 'gamma':
            # plot gamma distribution
            x = np.linspace(0, 10, 100)
            y = gamma.pdf(x, a=self.parameters[0], scale=self.parameters[1])
            axes[0].plot(x, y)
            axes[0].set_ylabel('Density')
            # plot hist of mu
            axes[1].hist(self.mu)
            axes[1].set_xlim(0, 10)
            axes[1].set_ylabel('Frequency')
        fig.suptitle('Environment')
        plt.tight_layout()
        plt.show()


In [ ]:
env = Environment(30, 'beta', 2, 2)
env.plot_distribution()


## Normative Algorithms

### **UGapEb: Upper Gap Elimination with Exploration Bonus** (Gabillon, Ghavamzadeh & Lazaric, 2012)

- Notation: 
  - $K$：number of arms
  - $T$：number of trials (budget)
  - $b(>0)$：exploration parameter

- Initialization: 
  - Pull each arm once and update $\hat{\mu}_k(K)$
  - Set $T_k(K)=1$
  
- Loop: 
  - Compute $B_k(t)=\max_{i\neq k}U_i(t) - L_k(t)$ for each arm $k \in \mathcal{A}$
    - Confidence interval: $\beta_k(t-1) = \sqrt{\frac{bT}{T_k(t-1)}}$
    - UCB (Upper confidence bound): $U_k(t)=\hat{\mu}_k(t-1) + \beta_k(t-1)$
    - LCB (Lower confidence bound): $L_k(t)=\hat{\mu}_k(t-1) - \beta_k(t-1)$
  - Identify the set of arms $J(t) \in \arg \min_{k \in \mathcal{A}} B_k(t)$
  - Pull the arm $I(t) = \arg \max_{k\in \{l_t,u_t\}} \beta_k(t − 1)$
    - $u_t = \arg\max_{j \notin J(t)}U_j(t)$
    - $l_t = \arg\min_{i \in J(t)}L_i(t)$
  - Observe $X_{I(t)}(T_{I(t)}(t − 1) + 1) \sim ν_{I(t)}$
  - Update $ \hat{\mu}_{I(t)}(t)$ and $T_{I(t)}(t)$


In [ ]:
class BaseAgent:
    def __init__(self, T: int, env: Environment, sigma: float):
        self.environment = env
        self.k = env.k # number of arms
        self.T = T # time horizon
        self.sigma = sigma # specificity of preference
        self.t = 0 # current time
        self.choice_frequency = np.zeros(self.k)
        self.cumulative_reward = np.zeros(self.k)
        self.mu_hat = np.zeros(self.k)
    
    def choose_arm(self):
        if np.min(self.choice_frequency) == 0:
            # there is an arm that has never been chosen
            arm = np.argmin(self.choice_frequency)
            return arm
        else:
            arm = np.random.choice(self.k)
        return arm
    
    def pull(self, arm: int):
        reward = self.environment.pull(arm) + np.random.normal(0, self.sigma)
        self.cumulative_reward[arm] += reward
        self.choice_frequency[arm] += 1
        self.mu_hat[arm] = self.cumulative_reward[arm] / self.choice_frequency[arm]
        self.t += 1
    
    def get_best_arm(self):
        arm = np.argmax(self.mu_hat)
        return arm
    
    def run(self):
        while self.t < self.T:
            arm = self.choose_arm()
            self.pull(arm)
    
    def get_result(self):
        res = {
            'N': self.T,
            'k': self.k,
            'sigma': self.sigma,
            'dist_parameters': self.environment.parameters,
            'true_mu': self.environment.mu.tolist(),
            'optimal_arm': np.argmax(self.environment.mu),
            'optimal_performance': np.max(self.environment.mu),
            'est_arm': self.get_best_arm(),
            'est_performance': self.environment.mu[self.get_best_arm()],
            'est_rank_within_K': np.argsort(self.environment.mu)[::-1].tolist().index(self.get_best_arm()) + 1,
            'is_successful': int(np.argsort(self.environment.mu)[::-1].tolist().index(self.get_best_arm()) == 0)
        }
        return res
    
    def print_mu_hat(self):
        print(self.mu_hat)
    
    def plot_choice_frequency(self):
        plt.bar(range(self.k), self.choice_frequency)
        # color the optimal arm
        plt.bar(np.argmax(self.environment.mu), self.choice_frequency[np.argmax(self.environment.mu)], color='red')
        plt.xlabel('Arm')
        plt.ylabel('Choice frequency')
        plt.title('Choice frequency of each arm')
        plt.show()
    
    def plot_mu_hat(self):
        plt.bar(range(self.k), self.mu_hat)
        plt.bar(np.argmax(self.environment.mu), self.mu_hat[np.argmax(self.environment.mu)], color='red')
        plt.scatter(range(self.k), self.environment.mu, color='black', marker='+', label='True value')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)
        plt.xlabel('Arm')
        plt.ylabel('Estimated mean')
        plt.title('Estimated mean of each arm')
        plt.show()


In [ ]:
class UGapEbAgent(BaseAgent):
    """
    UGapE algorithm
    Args:
        T: int
            time horizon
        env: Environment
            environment of the bandit problem
        sigma: float
            specificity of preference
        b: float
            hyper-parameter of UGapE algorithm (exploration rate)
    """
    def __init__(self, T: int, env: Environment, sigma: float, b: float):
        super().__init__(T = T, env = env, sigma = sigma)
        self.b = b

    def _compute_B(self, UCB, LCB):
        # B[i] = max_{j != i} UCB[j] - LCB[i], computed in O(k) instead of O(k^2).
        # For every arm except the current argmax of UCB, the leave-one-out max
        # equals the global max; for the argmax itself, it equals the runner-up.
        top1_idx = np.argmax(UCB)
        top1_val = UCB[top1_idx]
        runner_up_val = np.max(np.delete(UCB, top1_idx))
        max_excl = np.full(self.k, top1_val)
        max_excl[top1_idx] = runner_up_val
        return max_excl - LCB

    def choose_arm(self):
        if np.min(self.choice_frequency) == 0:
            # there is an arm that has never been chosen
            arm = np.argmin(self.choice_frequency)
            return arm
        else:
            # UGapEb algorithm
            beta = np.sqrt(self.b * self.T / self.choice_frequency)
            LCB = self.mu_hat - beta
            UCB = self.mu_hat + beta
            B = self._compute_B(UCB, LCB)

            # set of arms with minimum B
            J = np.where(B == np.min(B))[0]
            # set of arms with non-minimum B
            J_ = np.setdiff1d(range(self.k), J)

            # arm with maximum lower confidence bound in J
            l = J[np.argmin(LCB[J])]
            # arm with minimum upper confidence bound in J'
            u = J_[np.argmax(UCB[J_])]
            # union of l and u (potentially optimal arm)
            lu = np.union1d(l, u)

            arm = lu[np.argmax(beta[lu])]
        return int(arm)

    def get_best_arm(self):
        beta = np.sqrt(self.b * self.T / self.choice_frequency)
        LCB = self.mu_hat - beta
        UCB = self.mu_hat + beta
        B = self._compute_B(UCB, LCB)
        J = np.where(B == np.min(B))[0]
        arm = J[np.argmin(B[J])]
        return int(arm)

    def get_result(self):
        res = super().get_result()
        res['b'] = self.b
        return res


## Heuristics

### Random


In [ ]:
class RandomAgent(BaseAgent):
    def __init__(self, T: int, env: Environment, sigma: float):
        super().__init__(T = T, env = env, sigma = sigma)
    
    def choose_arm(self):
        arm = np.random.choice(self.k)
        return arm


### Exhaustive Search

In [ ]:
class ExhaustiveAgent(BaseAgent):
    def __init__(self, T: int, env: Environment, sigma: float):
        super().__init__(T = T, env = env, sigma = sigma)
    
    def choose_arm(self):
        arm = self.t % self.k
        return arm


### Greedy

In [ ]:
class GreedyAgent(BaseAgent):
    def __init__(self, T: int, env: Environment, sigma: float):
        super().__init__(T = T, env = env, sigma = sigma)
    
    def choose_arm(self):
        if np.min(self.choice_frequency) == 0:
            # there is an arm that has never been chosen
            arm = np.argmin(self.choice_frequency)
            return arm
        else:
            arm = np.argmax(self.mu_hat)
        return arm


## Simulation


Simulation reported in the paper (Figure 2): for each of 10,000 independently generated pools of 40 options ($\mu_i \sim \mathrm{Beta}(2,2)$), a random subset of $k$ options is drawn for each $k = 2, \dots, 40$, and the UGapEb policy is run on that subset with $T = 40$, $\sigma = 0.5$, and $b = 0.001$ (random seed 777).

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# Subset environment (uses only fixed mu values)
class _SubsetEnv:
    def __init__(self, mu):
        self.mu = np.array(mu)
        self.k = len(self.mu)
        self.type = "subset"
        self.parameters = None

    def pull(self, arm: int):
        if arm < 0 or arm >= self.k:
            raise ValueError("Invalid arm")
        return self.mu[arm]

np.random.seed(777)

iteration = 10000
T = 40
K_total = 40
dist = "beta"
b = 0.001

alpha = 2
beta = 2
sigma = 0.5

result = []
for _ in tqdm(range(iteration), desc="beta(2,2), sigma=0.5"):
    # 1) Generate a pool of 40 options
    base_env = Environment(K_total, dist, alpha, beta)
    total_mu = base_env.mu

    # 2) Draw k options and run them in the subset environment
    for k_ in range(2, 41, 1):
        subset_idx = np.random.choice(K_total, size=k_, replace=False)
        mu_subset = total_mu[subset_idx]
        env_subset = _SubsetEnv(mu_subset)

        agent = UGapEbAgent(T, env_subset, sigma, b)
        agent.run()
        res = agent.get_result()
        result.append(res)

filename = f"data_ugapeb_subset_T{T}_K{K_total}_{dist}_{alpha}_{beta}_sigma{sigma}_b{b}.pkl"

pd.to_pickle(result, filename)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


# ===== Settings =====
alpha, beta_param = 2, 2
T = 40
b = 0.001
sigma = 0.5
K_total = 40
dist = "beta"
fp = f"data_ugapeb_subset_T{T}_K{K_total}_{dist}_{alpha}_{beta_param}_sigma{sigma}_b{b}.pkl"

# ---- Load data ----
d = pd.DataFrame(pd.read_pickle(fp))
# est_performance is stored as a 1-element array per row; flatten to a plain float
d['est_performance'] = d['est_performance'].apply(lambda x: float(np.asarray(x).ravel()[0]))

# ---- Mean by k (achieved performance & best within subset) ----
temp = (
    d.groupby('k', as_index=False)
     .agg(
         chosen_true_mean=('est_performance', 'mean'),        # true value of the chosen arm
         ideal_subset_mean=('optimal_performance', 'mean')    # best within subset (dotted line)
     )
     .sort_values('k')
)

# ---- Compute the maximum point (k*) ----
idx_max = int(temp['chosen_true_mean'].values.argmax())
k_star = int(temp.iloc[idx_max]['k'])
y_star = float(temp.iloc[idx_max]['chosen_true_mean'])

# ---- Overall best (subset-best at k=40 = best among all candidates) ----
global_best = float(temp.loc[temp['k'] == K_total, 'ideal_subset_mean'].iloc[0])
# ---- Subset-best at k* (local best) ----
local_best_at_kstar = float(temp.loc[temp['k'] == k_star, 'ideal_subset_mean'].iloc[0])

print(f"[Beta(2,2), σ=0.5] k* = {k_star}, mean chosen_true = {y_star:.6f}")
print(f"global_best = {global_best:.6f}, local_best_at_kstar = {local_best_at_kstar:.6f}")

# ===== Plot =====
fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.5))

# Average payoff (true value of the final choice)
ax.plot(
    temp['k'], temp['chosen_true_mean'],
    color='black', linewidth=2,
    label='Obtained by the final choice'
)

# Maximum point (x) and vertical line
ax.scatter([k_star], [y_star], color='black', marker='x', s=120, zorder=5)
ax.axvline(k_star, color='black', linestyle=':', linewidth=1)
ax.text(
    k_star, y_star,
    f"  k*={k_star}\n  μ={y_star:.3f}",
    va='bottom', ha='left', fontsize=12, fontweight='bold', color='black'
)

# Best within subset (dotted line)
ax.plot(
    temp['k'], temp['ideal_subset_mean'],
    color='black', linestyle='dotted', linewidth=2,
    label='Best option in search set'
)

# ---- Overall best (red horizontal dashed line) ----
ax.axhline(
    global_best, color='red', linestyle='--', linewidth=1.5,
    label='Best option overall'
)

# ---- breadth-deficit loss: overall best <-> best in search set (orange, double arrow) ----
ax.annotate(
    '', xy=(k_star, local_best_at_kstar), xytext=(k_star, global_best),
    arrowprops=dict(arrowstyle='<->', color='orange', linewidth=2)
)
# ---- depth-deficit loss: best in search set <-> obtained payoff (blue, double arrow) ----
ax.annotate(
    '', xy=(k_star, y_star), xytext=(k_star, local_best_at_kstar),
    arrowprops=dict(arrowstyle='<->', color='steelblue', linewidth=2)
)

# ---- Appearance (text bold and larger for readability) ----
ax.set_title(r'$\mu_i \sim \mathrm{Beta}(2,2)$', fontsize=16, fontweight='bold', color='black')
ax.set_xlabel('Search-set', fontsize=16, fontweight='bold', color='black')
ax.set_ylabel('Expected payoff', fontsize=16, fontweight='bold', color='black')
ax.tick_params(axis='both', labelsize=12)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xlim(2, 40)
ax.set_xticks(np.arange(2, 41, 2))
ax.grid(True, axis='y', alpha=0.25)

leg = ax.legend(frameon=False, fontsize=12)
for text in leg.get_texts():
    text.set_fontweight('bold')
    text.set_color('black')

plt.tight_layout()
os.makedirs("img", exist_ok=True)
plt.savefig("img/fig_beta22_sigma05_only.png", dpi=300)
plt.show()
